In [1]:
import sys
import argostranslate.package
import argostranslate.translate
import json
import pandas as pd
from pathlib import Path

# ========================================
# 1. INSTALLATION (à exécuter UNE SEULE FOIS)
# ========================================

def install_translation_package(from_code="fr", to_code="en"):
    """
    Télécharge et installe le package de traduction.
    À exécuter une seule fois par paire de langues.
    """
    # Vérifie si déjà installé
    installed_packages = argostranslate.package.get_installed_packages()
    is_installed = any(
        pkg.from_code == from_code and pkg.to_code == to_code 
        for pkg in installed_packages
    )
    
    if is_installed:
        print(f"✅ Package {from_code} → {to_code} déjà installé")
        return True
    
    # Télécharge et installe
    try:
        print(f"📦 Téléchargement package {from_code} → {to_code}...")
        argostranslate.package.update_package_index()
        available_packages = argostranslate.package.get_available_packages()
        
        package_to_install = next(
            (pkg for pkg in available_packages 
             if pkg.from_code == from_code and pkg.to_code == to_code),
            None
        )
        
        if package_to_install is None:
            print(f"❌ Package {from_code} → {to_code} introuvable")
            return False
        
        download_path = package_to_install.download()
        argostranslate.package.install_from_path(download_path)
        print(f"✅ Installation terminée : {from_code} → {to_code}")
        return True
        
    except Exception as e:
        print(f"❌ Erreur lors de l'installation : {e}")
        return False


# ========================================
# 2. TRADUCTION SIMPLE
# ========================================

def translate_text(text, from_code="fr", to_code="en"):
    """
    Traduit un texte du français vers l'anglais.
    Le package doit être installé au préalable.
    """
    try:
        translated = argostranslate.translate.translate(text, from_code, to_code)
        return translated
    except Exception as e:
        print(f"❌ Erreur de traduction : {e}")
        return None


# ========================================
# 3. TRADUCTION AVEC CACHE (optimisé)
# ========================================

CACHE_FILE = Path("translation_cache_fr_en.json")

def load_cache():
    """Charge le cache de traductions"""
    if CACHE_FILE.exists():
        return json.loads(CACHE_FILE.read_text(encoding='utf-8'))
    return {}

def save_cache(cache):
    """Sauvegarde le cache"""
    CACHE_FILE.write_text(
        json.dumps(cache, ensure_ascii=False, indent=2), 
        encoding='utf-8'
    )

def translate_cached(text, from_code="fr", to_code="en"):
    """
    Traduit avec mise en cache locale.
    Évite de retraduire les mêmes textes.
    """
    cache = load_cache()
    key = f"{from_code}_{to_code}_{text}"
    
    # Vérifie le cache
    if key in cache:
        print(f"📦 Cache : '{text[:30]}...' → '{cache[key][:30]}...'")
        return cache[key]
    
    # Traduit et met en cache
    print(f"🌐 Traduction : '{text[:50]}...'")
    result = translate_text(text, from_code, to_code)
    
    if result:
        cache[key] = result
        save_cache(cache)
    
    return result


# ========================================
# 4. TRADUCTION PAR LOT
# ========================================

def translate_batch(texts, from_code="fr", to_code="en", use_cache=True):
    """
    Traduit une liste de textes du français vers l'anglais.
    """
    translate_func = translate_cached if use_cache else translate_text
    
    results = []
    for i, text in enumerate(texts, 1):
        print(f"[{i}/{len(texts)}] ", end="")
        translated = translate_func(text, from_code, to_code)
        results.append(translated)
    
    return results


# ========================================
# 5. UTILISATION / EXEMPLES
# ========================================

if __name__ == "__main__":
    
    # ÉTAPE 1 : Installation FR → EN (une seule fois)
    print("=" * 60)
    print("INSTALLATION PACKAGE FR → EN")
    print("=" * 60)
    install_translation_package("fr", "en")
    
    # ÉTAPE 2 : Traduction simple
    print("\n" + "=" * 60)
    print("TRADUCTION SIMPLE")
    print("=" * 60)
    
    texte_fr = "Bonjour le monde"
    resultat = translate_text(texte_fr, "fr", "en")
    print(f"FR : {texte_fr}")
    print(f"EN : {resultat}")
    
    # ÉTAPE 3 : Traduction avec cache
    print("\n" + "=" * 60)
    print("TRADUCTION AVEC CACHE")
    print("=" * 60)
    
    texte_fr2 = "Comment allez-vous ?"
    result1 = translate_cached(texte_fr2)  # 1ère fois
    result2 = translate_cached(texte_fr2)  # 2ème fois (depuis cache)
    
    # ÉTAPE 4 : Traduction par lot
    print("\n" + "=" * 60)
    print("TRADUCTION PAR LOT")
    print("=" * 60)
    
    textes_francais = [
        "Bonjour",
        "Merci beaucoup",
        "Bonne journée",
        "Au revoir",
        "Comment ça va ?",
        "Je ne comprends pas",
        "Pouvez-vous m'aider ?"
    ]
    
    traductions = translate_batch(textes_francais, "fr", "en", use_cache=True)
    
    print("\n📋 RÉSULTATS :")
    print("-" * 60)
    for fr, en in zip(textes_francais, traductions):
        print(f"FR : {fr:30} → EN : {en}")


C:\Users\kali\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INSTALLATION PACKAGE FR → EN
✅ Package fr → en déjà installé

TRADUCTION SIMPLE


2026-02-11 17:50:15 WARNING: Language fr package default expects mwt, which has been added


FR : Bonjour le monde
EN : Hello the world

TRADUCTION AVEC CACHE
📦 Cache : 'Comment allez-vous ?...' → 'How are you?...'
📦 Cache : 'Comment allez-vous ?...' → 'How are you?...'

TRADUCTION PAR LOT
[1/7] 📦 Cache : 'Bonjour...' → 'Hello....'
[2/7] 📦 Cache : 'Merci beaucoup...' → 'Thank you so much....'
[3/7] 📦 Cache : 'Bonne journée...' → 'Good day....'
[4/7] 📦 Cache : 'Au revoir...' → 'Goodbye....'
[5/7] 📦 Cache : 'Comment ça va ?...' → 'How are you?...'
[6/7] 📦 Cache : 'Je ne comprends pas...' → 'I don't understand....'
[7/7] 📦 Cache : 'Pouvez-vous m'aider ?...' → 'Can you help me?...'

📋 RÉSULTATS :
------------------------------------------------------------
FR : Bonjour                        → EN : Hello.
FR : Merci beaucoup                 → EN : Thank you so much.
FR : Bonne journée                  → EN : Good day.
FR : Au revoir                      → EN : Goodbye.
FR : Comment ça va ?                → EN : How are you?
FR : Je ne comprends pas            → EN : I don't unders

In [2]:
install_translation_package("fr", "en")


✅ Package fr → en déjà installé


True

In [3]:
resultat = translate_text("Bonjour le monde", "fr", "en")
print(resultat)  # Hello world


Hello the world


In [4]:
import pandas as pd
import re

# Charger le fichier CSV avec séparateur point-virgule
df = pd.read_csv(
    'prices_json_converted.csv',
    encoding='utf-8',
    sep=';',
    on_bad_lines='skip',
    engine='python'
)

print(f"✓ Fichier chargé: {len(df):,} lignes, {len(df.columns)} colonnes")
print(f"\nColonnes disponibles: {list(df.columns[:10])}...")

# Aperçu des données
df.head()


✓ Fichier chargé: 181,734 lignes, 33 colonnes

Colonnes disponibles: ['category_tag', 'created', 'currency', 'date', 'discount_type', 'duplicate_of', 'id', 'labels_tags', 'location', 'location_id']...


,category_tag,created,currency,date,discount_type,duplicate_of,id,labels_tags,location,location_id,...,proof,proof_id,receipt_quantity,source,tags,type,updated,Unnamed: 30,Unnamed: 31,Unnamed: 32
0,NaN,2024-10-28T16:55:51.817735Z,EUR,28/10/2024,NaN,NaN,42629,NaN,40,40,...,12176,12176.0,1.000,Smoothie - OpenFoodFacts (0.0.0+734),[],PRODUCT,2025-09-09T10:24:56.919108Z,NaN,NaN,NaN
1,NaN,2024-10-28T16:55:51.996910Z,EUR,28/10/2024,NaN,NaN,42630,NaN,40,40,...,12176,12176.0,1.000,Smoothie - OpenFoodFacts (0.0.0+734),[],PRODUCT,2025-09-09T10:24:56.962124Z,NaN,NaN,NaN
2,NaN,2024-10-19T08:13:22.166228Z,EUR,19/10/2024,NaN,NaN,39637,NaN,1263,1263,...,11463,11463.0,NaN,API,[],PRODUCT,2025-09-09T10:25:19.990971Z,NaN,NaN,NaN
3,NaN,2024-10-18T16:59:35.209751Z,EUR,18/10/2024,NaN,NaN,39573,NaN,11,11,...,11420,11420.0,NaN,Open Prices Web App,[],PRODUCT,2025-09-09T10:26:02.328383Z,NaN,NaN,NaN
4,NaN,2024-09-13T16:57:50.974292Z,EUR,10/09/2024,NaN,NaN,33784,NaN,872,872,...,8826,8826.0,1.000,Open Prices Web App,[],PRODUCT,2025-09-09T09:28:10.019273Z,NaN,NaN,NaN


In [5]:
print("=== NETTOYAGE ===\n")
print(f"Avant nettoyage: {len(df):,} lignes")

# Identifier les lignes vides (aucune info produit)
lignes_vides = (
    df['product_name'].isna() & 
    df['category_tag'].isna() & 
    df['product_code'].isna()
)

print(f"Lignes vides détectées: {lignes_vides.sum():,}")

# Supprimer et réinitialiser l'index
df = df[~lignes_vides].copy().reset_index(drop=True)

print(f"Après nettoyage: {len(df):,} lignes")
print(f"✓ {lignes_vides.sum():,} lignes supprimées")


=== NETTOYAGE ===

Avant nettoyage: 181,734 lignes
Lignes vides détectées: 0
Après nettoyage: 181,734 lignes
✓ 0 lignes supprimées


In [6]:
def extract_name_from_tag(tag):
    """Extrait le nom depuis un tag 'en:product-name'"""
    if pd.isna(tag):
        return None
    match = re.search('en:([a-zA-Z0-9-]+)', str(tag))
    if match:
        return match.group(1).replace('-', ' ').capitalize()
    return None

# Sauvegarder l'état original
df['product_name_original'] = df['product_name'].copy()

# Identifier les produits sans nom
null_mask = df['product_name'].isna()
print(f"Produits sans nom: {null_mask.sum():,}")

# Extraire depuis category_tag
df.loc[null_mask, 'extracted'] = df.loc[null_mask, 'category_tag'].apply(extract_name_from_tag)

# Imputer
impute_mask = null_mask & df['extracted'].notna()
df.loc[impute_mask, 'product_name'] = df.loc[impute_mask, 'extracted']

# Tracer la source
df['imputation_source'] = 'original'
df.loc[impute_mask, 'imputation_source'] = 'imputed_from_category'

print(f"✓ {impute_mask.sum():,} produits imputés")

# Exemples
print("\nExemples de produits imputés:")
df[impute_mask][['product_name', 'category_tag', 'price', 'price_per']].head(10)


Produits sans nom: 77,914
✓ 4,105 produits imputés

Exemples de produits imputés:


,product_name,category_tag,price,price_per
199,Broccoli,en:broccoli,3.90,KILOGRAM
204,Garlic,en:garlic,14.95,KILOGRAM
234,Bananas,en:bananas,1.99,KILOGRAM
235,Carrots,en:carrots,2.50,KILOGRAM
284,Chestnuts,en:chestnuts,29.90,KILOGRAM
290,Baguettes,en:baguettes,4.35,KILOGRAM
325,Cucumbers,en:cucumbers,5.98,KILOGRAM
562,Lemons,en:lemons,5.20,KILOGRAM
575,Traditional french baguette,en:traditional-french-baguette,1.00,UNIT
576,Bananas,en:bananas,1.99,KILOGRAM


In [7]:
import argostranslate.package
import argostranslate.translate

print("Vérification du modèle de traduction...")

# Vérifier si déjà installé
installed = argostranslate.translate.get_installed_languages()
model_installed = (
    any(lang.code == "fr" for lang in installed) and 
    any(lang.code == "en" for lang in installed)
)

if not model_installed:
    print("Téléchargement du modèle FR → EN (~100 MB)...")
    argostranslate.package.update_package_index()
    available = argostranslate.package.get_available_packages()
    package = next(filter(lambda x: x.from_code == "fr" and x.to_code == "en", available))
    argostranslate.package.install_from_path(package.download())
    print("✓ Modèle installé")
else:
    print("✓ Modèle déjà installé")


Vérification du modèle de traduction...
✓ Modèle déjà installé


In [8]:
# Configurer la traduction
installed = argostranslate.translate.get_installed_languages()
from_lang = next(filter(lambda x: x.code == "fr", installed))
to_lang = next(filter(lambda x: x.code == "en", installed))
translation = from_lang.get_translation(to_lang)

# Traduire tous les produits avec un nom
print(f"Traduction de {df['product_name'].notna().sum():,} produits...")

df['product_name_en'] = df['product_name'].apply(
    lambda x: translation.translate(str(x)) if pd.notna(x) else None
)

print(f"✓ {df['product_name_en'].notna().sum():,} produits traduits")

# Exemples
print("\nExemples de traductions:")
df[['product_name', 'product_name_en', 'imputation_source']].head(15)


Traduction de 107,925 produits...
✓ 107,925 produits traduits

Exemples de traductions:


,product_name,product_name_en,imputation_source
0,NaN,None,original
1,NaN,None,original
2,NaN,None,original
3,NaN,None,original
4,NaN,None,original
5,NaN,None,original
6,NaN,None,original
7,NaN,None,original
8,NaN,None,original
9,NaN,None,original


In [9]:
output_file = 'prices_cleaned_imputed_translated.csv'
df.to_csv(output_file, index=False, encoding='utf-8', sep=';')

print(f"✓ Fichier sauvegardé: {output_file}")
print(f"\n=== STATISTIQUES FINALES ===")
print(f"Total lignes: {len(df):,}")
print(f"Produits imputés: {(df['imputation_source'] == 'imputed_from_category').sum():,}")
print(f"Produits originaux: {(df['imputation_source'] == 'original').sum():,}")
print(f"Produits traduits: {df['product_name_en'].notna().sum():,}")
print(f"Produits toujours NULL: {df['product_name'].isna().sum():,}")


✓ Fichier sauvegardé: prices_cleaned_imputed_translated.csv

=== STATISTIQUES FINALES ===
Total lignes: 181,734
Produits imputés: 4,105
Produits originaux: 177,629
Produits traduits: 107,925
Produits toujours NULL: 73,809
